# 10b — Ternary Error Analysis

This notebook analyzes BanglaSarc3 ternary:
- plain BanglaBERT
- weighted/confusion-aware BanglaBERT

It focuses on:
- confusion matrices
- hardest class pairs
- cases fixed by weighted training
- cases worsened by weighted training

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from sklearn.metrics import confusion_matrix

TABLES = Path("../04_outputs/tables")
SPLITS = Path("../01_data/interim/splits")
CHECKPOINTS = Path("../03_models/checkpoints")
MODEL_NAME = "csebuetnlp/banglabert"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

def resolve_checkpoint_dir(checkpoint_root):
    checkpoint_root = Path(checkpoint_root)
    if (checkpoint_root / "config.json").exists():
        return checkpoint_root
    ckpts = sorted(
        [p for p in checkpoint_root.glob("checkpoint-*") if p.is_dir()],
        key=lambda p: int(p.name.split("-")[-1])
    )
    if not ckpts:
        raise FileNotFoundError(f"No checkpoint-* folder found inside: {checkpoint_root}")
    trainer_state_file = checkpoint_root / "trainer_state.json"
    if trainer_state_file.exists():
        with open(trainer_state_file, "r", encoding="utf-8") as f:
            trainer_state = json.load(f)
        best_ckpt = trainer_state.get("best_model_checkpoint", None)
        if best_ckpt:
            best_ckpt = Path(best_ckpt)
            if best_ckpt.exists():
                return best_ckpt
    return ckpts[-1]

/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
TERNARY_LABELS = {0: "Non-Sarcastic", 1: "Neutral", 2: "Sarcastic"}

def load_ternary_predictions(checkpoint_root, split_file):
    df = pd.read_csv(split_file).copy()
    ds_df = df[["text", "label_ternary"]].rename(columns={"label_ternary": "label"})
    ds = Dataset.from_pandas(ds_df, preserve_index=False)
    ds = ds.map(tokenize_batch, batched=True)
    ds = ds.remove_columns(["text"])
    ds.set_format("torch")

    checkpoint_dir = resolve_checkpoint_dir(checkpoint_root)
    print("Loading:", checkpoint_dir)

    model = AutoModelForSequenceClassification.from_pretrained(checkpoint_dir)
    trainer = Trainer(model=model)
    output = trainer.predict(ds)
    logits = output.predictions
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds = np.argmax(probs, axis=1)

    out = df.copy()
    out["y_true"] = df["label_ternary"].astype(int).to_numpy()
    out["y_pred"] = preds
    out["prob_0"] = probs[:, 0]
    out["prob_1"] = probs[:, 1]
    out["prob_2"] = probs[:, 2]
    out["correct"] = out["y_true"] == out["y_pred"]
    out["pred_conf"] = [probs[i, preds[i]] for i in range(len(preds))]
    return out

In [3]:
def ternary_confusions(df):
    cm = confusion_matrix(df["y_true"], df["y_pred"], labels=[0,1,2])
    return pd.DataFrame(cm, index=[TERNARY_LABELS[i] for i in [0,1,2]], columns=[TERNARY_LABELS[i] for i in [0,1,2]])

def confusion_examples(df, true_label, pred_label, n=20):
    sub = df[(df["y_true"] == true_label) & (df["y_pred"] == pred_label)].copy()
    return sub.sort_values("pred_conf", ascending=False)[
        ["text", "y_true", "y_pred", "prob_0", "prob_1", "prob_2", "pred_conf"]
    ].head(n)

def compare_models(base_df, improved_df, n=20):
    merged = pd.DataFrame({
        "text": base_df["text"],
        "y_true": base_df["y_true"],
        "base_pred": base_df["y_pred"],
        "base_correct": base_df["correct"],
        "improved_pred": improved_df["y_pred"],
        "improved_correct": improved_df["correct"],
    })
    improved_cases = merged[(merged["base_correct"] == False) & (merged["improved_correct"] == True)].copy()
    degraded_cases = merged[(merged["base_correct"] == True) & (merged["improved_correct"] == False)].copy()
    return improved_cases.head(n), degraded_cases.head(n)

In [4]:
plain_df = load_ternary_predictions(
    "../03_models/checkpoints/banglabert_banglasarc3_ternary",
    "../01_data/interim/splits/banglasarc3_ternary_test.csv",
)
weighted_df = load_ternary_predictions(
    "../03_models/checkpoints/banglabert_weighted_banglasarc3_ternary",
    "../01_data/interim/splits/banglasarc3_ternary_test.csv",
)

print("Plain confusion matrix")
display(ternary_confusions(plain_df))

print("Weighted confusion matrix")
display(ternary_confusions(weighted_df))

Map: 100%|██████████| 1208/1208 [00:00<00:00, 17632.39 examples/s]


Loading: ../03_models/checkpoints/banglabert_banglasarc3_ternary/checkpoint-2416


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8912.73it/s]
/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Map: 100%|██████████| 1208/1208 [00:00<00:00, 18101.44 examples/s]


Loading: ../03_models/checkpoints/banglabert_weighted_banglasarc3_ternary/checkpoint-2416


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8421.62it/s]
/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Plain confusion matrix


,Non-Sarcastic,Neutral,Sarcastic
Non-Sarcastic,220,99,82
Neutral,92,275,39
Sarcastic,86,35,280


Weighted confusion matrix


,Non-Sarcastic,Neutral,Sarcastic
Non-Sarcastic,238,89,74
Neutral,90,271,45
Sarcastic,89,33,279


In [5]:
print("Plain: Non-Sarcastic -> Neutral")
display(confusion_examples(plain_df, 0, 1, n=15))

print("Plain: Non-Sarcastic -> Sarcastic")
display(confusion_examples(plain_df, 0, 2, n=15))

print("Plain: Neutral -> Sarcastic")
display(confusion_examples(plain_df, 1, 2, n=15))

print("Plain: Sarcastic -> Neutral")
display(confusion_examples(plain_df, 2, 1, n=15))

Plain: Non-Sarcastic -> Neutral


,text,y_true,y_pred,prob_0,prob_1,prob_2,pred_conf
583,চিন্তা বোধ শক্তির কেন্দ্র উৎপত্তি অবস্থান মস্ত...,0,1,0.052093,0.937736,0.010172,0.937736
339,ডিপ্লোমা ইঞ্জিনিয়ার যারা তাদের কাছে একটা প্রশ্...,0,1,0.046582,0.935964,0.017454,0.935964
469,উভয়েরই রোষানলের আগুনে পুড়া ছাই পাওয়া যাবে বটে ...,0,1,0.052860,0.935589,0.011551,0.935589
230,লেখাটা অনেক গভীর বাস্তবতাকে খুব সুন্দরভাবে তুল...,0,1,0.070581,0.918817,0.010602,0.918817
242,সমানুপাতিক যেহেতু যেকোনো একটা কমান তাহলে অন্যট...,0,1,0.070325,0.911771,0.017905,0.911771
1129,সম্পর্কের ভবিষ্যৎ নির্ধারণে যৌথ সিদ্ধান্ত নেওয়...,0,1,0.080911,0.910413,0.008676,0.910413
639,একটি স্টার্টআপের টিকে থাকার জন্য ভালো পরিকল্পন...,0,1,0.088899,0.902415,0.008686,0.902415
503,প্রত্যেক মানুষের জীবনযাত্রা আলাদা কেউ আগে শেখে...,0,1,0.086120,0.901625,0.012255,0.901625
93,রোগীর মা র সত্যি কথা বলা চিকিৎসকদের সঠিক চিকিৎ...,0,1,0.101401,0.889403,0.009197,0.889403
371,গুম কমিশনের উচিত এগুলো ভিডিও আকারে ডকুমেন্ট কর...,0,1,0.098237,0.887549,0.014214,0.887549


Plain: Non-Sarcastic -> Sarcastic


,text,y_true,y_pred,prob_0,prob_1,prob_2,pred_conf
1194,এই নে আম্নে গো কলেজের দৃশ্য,0,2,0.058090,0.021970,0.919939,0.919939
464,এরা এখন সেহেরি পারটি করতে পারেনা,0,2,0.050303,0.030053,0.919644,0.919644
441,হেফাজত মানে সেক্স করা হা হা হা৷,0,2,0.071383,0.013925,0.914692,0.914692
266,আর যাবোনা বেগুন তুলিতে,0,2,0.068795,0.018847,0.912358,0.912358
798,ভাই আপনি এটা কোনো কাম করলেন হাসতেও পারতেছিনা ব...,0,2,0.065466,0.024140,0.910394,0.910394
990,হাসিনা আর কাদের মামুকে জাদুঘর দেখাশোনার দায়িত্...,0,2,0.075233,0.025845,0.898921,0.898921
223,কাচ্চি ডাইন এর কি বিজ্ঞাপনের খরা লেগেছে,0,2,0.088787,0.014308,0.896905,0.896905
118,শুধু মাঝে মাঝে বলতো ঘুমা মোবাইল না টিপে,0,2,0.078661,0.032117,0.889223,0.889223
611,তামিমের চেহারা অনেকটা ইলিয়াস কাঞ্চনের মত হয়েছে...,0,2,0.088811,0.024456,0.886732,0.886732
278,এদেহি সেম আমার মত সাহস হ্যা,0,2,0.095084,0.018494,0.886422,0.886422


Plain: Neutral -> Sarcastic


,text,y_true,y_pred,prob_0,prob_1,prob_2,pred_conf
816,পায়ের জুতাটা অটোমেটিক খুলে গেলো,1,2,0.054228,0.019838,0.925935,0.925935
390,সকাল সকাল খালি গায় আপনার ঠান্ডা লাগে না৷,1,2,0.071074,0.016678,0.912248,0.912248
1083,হাতিয়া নাকি এখন আওয়ামী লীগের দখলে,1,2,0.094322,0.023838,0.881840,0.881840
893,আম া দের প্রধানমন্ত্রী মাদার অব হিউ মে নি টি জ...,1,2,0.102140,0.023160,0.874700,0.874700
604,আসলে ইনি কে দেখলাম মেলায় গোলাপ দিচ্ছে,1,2,0.094030,0.031424,0.874546,0.874546
534,ভাজ্ঞিস এসব স্থানে আমি থাকি না,1,2,0.089335,0.039608,0.871057,0.871057
532,আমার প্রিয় মানুষ টা কে ফুল দেওয়ার দরকার নাই কা...,1,2,0.103801,0.033479,0.862720,0.862720
696,মুরগির চিকেন মানলাম কিন্তু নিপু ভাই টমাটোম কি,1,2,0.074644,0.063218,0.862138,0.862138
322,বিষটি তাড়াতাড়ি আটকাতে হবে নয়তো আজ অন্য রোবট...,1,2,0.126216,0.021293,0.852491,0.852491
472,নাম্বার টা আমার বর ভাইয়ের অবস্থান একজন প্রবাস...,1,2,0.114918,0.035224,0.849858,0.849858


Plain: Sarcastic -> Neutral


,text,y_true,y_pred,prob_0,prob_1,prob_2,pred_conf
408,দাঁড়ি কেটে ফেলায় সালমান এফ রহমান কে যখন চেনা য...,2,1,0.053816,0.933947,0.012236,0.933947
68,এসব ইনজোরিতে ফিডনেস আরো ভালো থাকে,2,1,0.104439,0.876535,0.019027,0.876535
1067,মোহাম্মদপুরে থেকে হারিয়ে যাওয়া কোনো জিনিস এই প...,2,1,0.085112,0.873056,0.041832,0.873056
692,মোহাম্মদপুরের ইতিহাসে এই প্রথম কোন হারানো জিনি...,2,1,0.116535,0.871162,0.012304,0.871162
728,আপনাকে তো শিক্ষিতই ভাবতাম পিনাকীর বাইসেপ কতটুক...,2,1,0.117683,0.850995,0.031322,0.850995
787,ওয়াও আবার কয়েক মিনিটের মধ্যেই একসাথে অনেকগুল...,2,1,0.126614,0.835061,0.038325,0.835061
807,অনেক গুরুত্বপূর্ণ একটা ক্লাস নিল ষাঁড়ে,2,1,0.131709,0.830639,0.037651,0.830639
1113,সঠিকভাবে লিখেছেন সেজন্য ধন্যবাদ খুব উপকার করলেন,2,1,0.154345,0.826894,0.018761,0.826894
652,সর্বস্তরের নেতাকর্মীবৃন্দ শেখ হাসিনার প্রশ্নে ...,2,1,0.140922,0.822907,0.036171,0.822907
715,প্রাইভেট পাওয়ার প্লান্ট নির্মাণ বন্ধ এবং সময় শ...,2,1,0.190633,0.796764,0.012603,0.796764


In [6]:
improved_cases, degraded_cases = compare_models(plain_df, weighted_df, n=20)

print("Cases fixed by weighted/confusion-aware model")
display(improved_cases)

print("Cases worsened by weighted/confusion-aware model")
display(degraded_cases)

Cases fixed by weighted/confusion-aware model


,text,y_true,base_pred,base_correct,improved_pred,improved_correct
30,হাদিসের আলোকে জীবন পরিচালনা করা খুবই জরুরি আল্...,0,1,False,0,True
64,এই সমস্যার সমাধান একমাত্র রিপন ভাই দিতে পারবে ...,2,0,False,2,True
82,গোলাপ ফুল দিয়ে পুজা হয়না,2,0,False,2,True
92,কি সুন্দর করে বললেন আমি পরকালে বিশ্বাস করি না ...,0,1,False,0,True
96,জীবন কে মানুষ কত সহজ ভাবে নিচ্ছে,0,1,False,0,True
99,ডাক্তাররা সর্বদা রোগীদের সেবা দিতে প্রস্তুত,0,1,False,0,True
100,এদের উপর নজর পরলো কেনো আবার,2,0,False,2,True
129,সস্তা পাবলিসিটি আর বিনোদন এর জন্য বড়আ কে ধন্য...,0,2,False,0,True
154,নিশো আর মেহজাবিন নাটক এর কথা মনে পড়ে গেলো নিশো...,2,0,False,2,True
156,মানুষ এত জ্ঞানী হয় ওনাকে না দেখলে বুজতাম না,1,0,False,1,True


Cases worsened by weighted/confusion-aware model


,text,y_true,base_pred,base_correct,improved_pred,improved_correct
109,শেখ হাসিনা একজন ই যার বিকল্প উনি নিজেই কাজেই ব...,2,2,True,0,False
117,লিডার পরবর্তী নেতা,2,2,True,0,False
126,ভুয়া কথা বিবাহিত রাও দেয় যারা বেকার তারা,2,2,True,0,False
139,ইউনুস সরকার দেশ পরিচালনা করতে ব্যর্থ তাই তার উ...,2,2,True,0,False
151,না রে ভাই সকালে এলার্ম বাজার শব্দ থেকেও ঘুমের ...,2,2,True,1,False
217,নতুন পুরাতন ইউজার ভাই ও বোনদের জন্য দোয়া রইলো ...,1,1,True,0,False
229,খাবারে ব্লেড পাওয়ায় আজ অথোরিটি ওটা তো ছোট ব্লে...,2,2,True,0,False
260,সে আবার বড় বড় কথা বলে এই গুলা আবার পাবলিক খায়,0,0,True,2,False
270,আপনার পোস্ট টা পড়ে মাত্র সার্চ করলাম সারাহ টেন...,1,1,True,2,False
290,মাশাল্লাহ সৃষ্টিকর্তা আমাদের মাঝে কত কিছু না দ...,1,1,True,0,False


In [7]:
summary = pd.DataFrame([
    {"model": "banglabert", "errors": int((~plain_df["correct"]).sum())},
    {"model": "banglabert_weighted", "errors": int((~weighted_df["correct"]).sum())},
])
summary["error_reduction_vs_plain"] = summary.loc[0, "errors"] - summary["errors"]
display(summary)
summary.to_csv(TABLES / "ternary_error_analysis_summary.csv", index=False)
print("Saved:", TABLES / "ternary_error_analysis_summary.csv")

,model,errors,error_reduction_vs_plain
0,banglabert,433,0
1,banglabert_weighted,420,13


Saved: ../04_outputs/tables/ternary_error_analysis_summary.csv
